In [ ]:
import os
import random
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm
import glob
from sklearn import cluster
from sklearn import metrics
from sklearn import model_selection
from sklearn import ensemble

import seaborn as sns

from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score


from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.feature_selection import mutual_info_classif

In [ ]:
# Read the data and append the Id
def read_data(path):
    df = pd.read_csv(path)
    df['Id'] = path.split("\\")[-1].split(".")[0]
    
    return df


# Read and concatenate all files of the train data from specified dataset
def create_full_data(dataset_name):
    paths = glob.glob(data_directory + f'train/{dataset_name}/*')
    final_df = pd.concat([read_data(p) for p in paths])
    final_df['dataset'] = dataset_name
    
    return final_df

In [ ]:
# Create a dataframe with all defog data
defog_data = create_full_data('defog')
defog_data.head()

In [ ]:
len(defog_data)

In [ ]:
tdcsfog_metadata_df.head()

In [ ]:
defog_metadata_df.head()

In [ ]:
subjects_df.head()

In [ ]:
# list of all tdcsfog csv file path
tdcs_file_paths = glob.glob(os.path.join(data_directory, 'train', 'tdcsfog', '*.csv').replace('\\', '/'), recursive=True)
tdcs_file_paths = [str(path).replace('\\', '/') for path in tdcs_file_paths]
print(f'the number of files to be read: {len(tdcs_file_paths)}')

In [ ]:
train_tdcsfog_example_df

In [ ]:
# Initialize a DataFrame to combine data from multiple CSV files.
tdcsfog_dfs = pd.DataFrame()

# load tdcsfog time series in combination with metadata.
for fp in tdcs_file_paths:    
    
    # load data into a variable 'tmp'.
    tmp = pd.read_csv(fp)
    
    # get file Id from csv file name.
    file_id = os.path.basename(fp).replace(".csv", "")

    # Add the Id to the tmp DataFrame
    tmp['Id'] = file_id

    
    # concat the data
    tdcsfog_dfs = pd.concat([tdcsfog_dfs, tmp]).reset_index(drop=True)

In [ ]:
tdcsfog_dfs.to_csv(data_directory+"train_tdcsfog_dfs.csv", index=False)

In [ ]:
#tdcsfog_dfs.to_csv(data_directory+"train/tdcsfog/tdcsfog_dfs.csv", index=False)
tdcsfog_dfs = pd.read_csv(data_directory+"train_tdcsfog_dfs.csv")

In [ ]:
# check the contents of the tdcsfog_dfs
tdcsfog_dfs.head()

In [ ]:
# list of all defog csv file path
defog_file_paths = glob.glob(os.path.join(data_directory, 'train', 'defog', '*.csv').replace('\\', '/'), recursive=True)
defog_file_paths = [str(path).replace('\\', '/') for path in defog_file_paths]


print(f'the number of files to be read: {len(defog_file_paths)}')

In [ ]:
# Initialize a DataFrame to combine data from multiple CSV files.
defog_dfs = pd.DataFrame()

for fp in defog_file_paths:
    # load data into a variable 'tmp'.
    tmp = pd.read_csv(fp)
    
    # get file Id from csv file name.
    file_id = os.path.basename(fp).replace(".csv", "")
    
    # Add the Id to the tmp DataFrame
    tmp['Id'] = file_id
    # get subject Id.
    subject = defog_metadata_df.loc[defog_metadata_df['Id'] == file_id, 'Subject'].iloc[0]
    
    
    # extract data from the time period where Valid and Task are both True.
    tmp = tmp[(tmp['Valid'] == True) & (tmp['Task']==True)]
    tmp = tmp.drop(['Valid', 'Task'], axis=1)
    
    # concat the data
    defog_dfs = pd.concat([defog_dfs, tmp]).reset_index(drop=True)

In [ ]:
# check the contents of the defog_dfs
defog_dfs.head()

In [ ]:
subjects_df

In [ ]:
defog_metadata_df

In [ ]:
# check the contents of the defog_dfs
defog_dfs['dataset']='defog'
defog_dfs.head()

#### Feature engineering

In [ ]:
len(pd.concat([tdcsfog_dfs, defog_dfs]))

In [ ]:
tasks_df['Duration'] = tasks_df['End'] - tasks_df['Begin']

tasks_pivot_df = pd.pivot_table(tasks_df, values=['Duration'],
                                index=['Id'], columns=['Task'],
                                aggfunc='sum', fill_value=0)

tasks_pivot_df.columns = [c[-1] for c in tasks_pivot_df.columns]
tasks_pivot_df = tasks_pivot_df.reset_index()

tasks_pivot_df['t_kmeans'] = KMeans(n_clusters=10, random_state=42, n_init=10) \
                    .fit_predict(tasks_pivot_df[tasks_pivot_df.columns[1:]])


In [ ]:
tasks_df

In [ ]:
tasks_pivot_df

In [ ]:
subjects_df

In [ ]:
subjects_df = subjects_df.fillna(0).groupby('Subject') \
    [['Visit', 'Age', 'YearsSinceDx', 'UPDRSIII_On', 'UPDRSIII_Off', 'NFOGQ']].median()
subjects_df = subjects_df.reset_index()

subjects_df['s_kmeans'] = KMeans(n_clusters=10, random_state=42, n_init=10) \
                    .fit_predict(subjects_df[subjects_df.columns[1:]])

In [ ]:
subjects_df

In [ ]:
train_data_df = pd.concat([tdcsfog_dfs, defog_dfs])
train_data_df.shape

In [ ]:
train_data_df.head()

In [ ]:
# Create multiclass event type column
train_data_df["target"] = "Normal"
targets = ["StartHesitation", "Turn", "Walking"]
for target in targets:
    train_data_df.loc[train_data_df[target] == 1, "target"] = target

train_data_df['Sex'] = train_data_df['Sex'].map({'M': 1, 'F': 0})
train_data_df['Medication'] = train_data_df['Medication'].map({'on': 1, 'off': 0})


# Keep only columns that we need
train_data_df.drop(["StartHesitation", "Turn", "Walking"],axis=1,inplace=True)

In [ ]:
train_data_df

In [ ]:
train_data_df.NFOGQ.isnull().sum()

In [ ]:
full_data = full_data.merge(full_metadata, on='Id', how='inner') \
                        .merge(tasks_data[['Id','t_kmeans']], how='left', on='Id').fillna(-1) \
                        .merge(subjects_data.drop('Visit', axis=1),
                                       on='Subject', how='left').fillna(-1)
full_data.head()

In [ ]:
full_data[full_data.dataset == 'defog'].index[0]

In [ ]:
full_data_sample = pd.concat([full_data[:1000_000], full_data[7_062_672:8_062_672]])


In [ ]:
full_data_sample.shape


In [ ]:
full_data

In [ ]:
# Create multiclass event type column
full_data_sample['target'] = 'Normal'
targets = ['StartHesitation', 'Turn', 'Walking']
for target in targets:
    full_data_sample.loc[full_data_sample[target] == 1, 'target'] = target
    
full_data_sample.Medication = np.where(full_data_sample.Medication == 'off', 0, 1)

In [ ]:
full_data_sample.head()

In [ ]:
full_data_sample_test = full_data_sample.copy()

In [ ]:
acc_cols = ['AccV', 'AccML', 'AccAP']
for acc in acc_cols:
        full_data_sample_test[f'{acc}_lag_2'] = full_data_sample_test.groupby('Id')[acc].shift(2).fillna(method="backfill")

full_data_sample_test.head()

In [ ]:
full_data_sample[full_data_sample.Id=='003f117e14']

In [ ]:
full_data_sample[full_data_sample.Id=='003f117e14']['AccV'].plot()

In [ ]:
full_data_sample[full_data_sample.Id=='003f117e14']['AccV_lag_2'].plot()

In [ ]:
full_data_sample[full_data_sample.Id=='003f117e14']['AccV_first_value']

In [ ]:
full_data_sample.columns

In [ ]:
# Create new features based on accelerometer data columns and returns the updated dataframe
def create_features(df):
    acc_cols = ['AccV', 'AccML', 'AccAP']

    for acc in acc_cols:
        df[f'{acc}_lag_2'] = df.groupby('Id')[acc].shift(2).fillna(method="backfill")
        df[f'{acc}_lag_3'] = df.groupby('Id')[acc].shift(3).fillna(method="backfill")
        df[f'{acc}_lag_4'] = df.groupby('Id')[acc].shift(4).fillna(method="backfill")
        df[f'{acc}_lag_5'] = df.groupby('Id')[acc].shift(5).fillna(method="backfill")

        df[f'{acc}_cumsum'] = (df[acc]).groupby(df['Id']).cumsum()

        df[f'{acc}_first_value'] = df.groupby('Id')[acc].transform('first')
        df[f'{acc}_last_value'] = df.groupby('Id')[acc].transform('last')

        df[f'{acc}_mean'] = df.groupby('Id')[acc].transform('mean')
        df[f'{acc}_median'] = df.groupby('Id')[acc].transform('median')
        df[f'{acc}_std'] = df.groupby('Id')[acc].transform('std')

        df[f'{acc}_min'] = df.groupby('Id')[acc].transform('min')
        df[f'{acc}_max'] = df.groupby('Id')[acc].transform('max')

        df[f'{acc}_delta'] = df[f'{acc}_max'] - df[f'{acc}_min']
        for lag in [1,2,3]:
            df[f'ma_{lag}_{acc}'] = df[acc].rolling(lag).mean().fillna(method="backfill")
            df[f'ma{lag}_{acc}'] = df[acc].te(lag).mean().fillna(method="backfill")
            df[f'ma_{lag}_{acc}'] = df[acc].rolling(lag).mean().fillna(method="backfill")
        
    return df

In [ ]:
full_data_sample.head(10)

In [ ]:
full_data_sample[full_data_sample.Id=='2e75cf4507']

In [ ]:
full_data_sample.Id.value_counts()

In [ ]:
# Apply the above function to our dataset
full_data_sample = create_features(full_data_sample)

In [ ]:
full_data_sample.head()

In [ ]:
full_data_sample.columns

In [ ]:
# Assign variables with predictors and target
X = full_data_sample.drop(['Id', 'dataset', 'Test', 'Subject', 'UPDRSIII_Off',
                               'StartHesitation', 'Turn', 'Walking'], axis=1).copy()
y = X.pop('target')

In [ ]:
# Check the missing values in our X dataset
X.isna().sum()[X.isna().sum() > 0]

In [ ]:
# Check the categorical variables in our X dataset
X.dtypes[X.dtypes == object]

In [ ]:
discrete_features = X.dtypes == int

# Calculate MI scores for the features in X with respect to the target variable y
def make_mi_scores(X, y, discrete_features):
    mi_scores = mutual_info_classif(X, y, discrete_features=discrete_features)
    mi_scores = pd.Series(mi_scores, name="MI Scores", index=X.columns)
    mi_scores = mi_scores.sort_values(ascending=False)
    return mi_scores

# Apply this function to our data
mi_scores = make_mi_scores(X, y, discrete_features)

In [ ]:
# Show 10 features with the highest MI score
mi_scores.to_frame().head(10)

In [ ]:
def plot_mi_scores(scores):
    scores = scores.sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(13, 11))
    ax = sns.barplot(x=scores.values, y=scores.index, palette="coolwarm", orient='h')
    ax.set_title("Mutual Information Scores")
    ax.set_xlabel("MI Scores")
    ax.set_ylabel("Features")
    plt.show()

plot_mi_scores(mi_scores)
plt.show()

In [ ]:
defog_data[defog_data.Id=='02ea782681'].AccV.plot()

In [ ]:
defog_data_valid[defog_data_valid.Id=='02ea782681'].AccV.plot()

In [ ]:
defog_data_valid.columns

In [ ]:
# Create new features based on accelerometer data columns and returns the updated dataframe
def create_features(df):
    acc_cols = ['AccV', 'AccML', 'AccAP']

    for acc in acc_cols:
        df[f'{acc}_lag_2'] = df.groupby('Id')[acc].shift(2).fillna(method="backfill")
        df[f'{acc}_lag_3'] = df.groupby('Id')[acc].shift(3).fillna(method="backfill")
        df[f'{acc}_lag_4'] = df.groupby('Id')[acc].shift(4).fillna(method="backfill")
        df[f'{acc}_lag_5'] = df.groupby('Id')[acc].shift(5).fillna(method="backfill")

        df[f'{acc}_cumsum'] = (df[acc]).groupby(df['Id']).cumsum()

        df[f'{acc}_first_value'] = df.groupby('Id')[acc].transform('first')
        df[f'{acc}_last_value'] = df.groupby('Id')[acc].transform('last')

        df[f'{acc}_mean'] = df.groupby('Id')[acc].transform('mean')
        df[f'{acc}_median'] = df.groupby('Id')[acc].transform('median')
        df[f'{acc}_std'] = df.groupby('Id')[acc].transform('std')

        df[f'{acc}_min'] = df.groupby('Id')[acc].transform('min')
        df[f'{acc}_max'] = df.groupby('Id')[acc].transform('max')

        df[f'{acc}_delta'] = df[f'{acc}_max'] - df[f'{acc}_min']
        for lag in [1,2,3]:
            df[f'ma_{lag}_{acc}'] = df[acc].rolling(lag).mean().fillna(method="backfill")
            df[f'ma{lag}_{acc}'] = df[acc].rolling(lag).mean().fillna(method="backfill")
            df[f'ma_{lag}_{acc}'] = df[acc].rolling(lag).mean().fillna(method="backfill")
        
    return df

In [ ]:
defog_data_valid2 = defog_data.loc[(defog_data.Valid == True) & (defog_data.Task == True)].copy()

defog_data_valid2.reset_index(drop=True, inplace=True)
defog_data_valid2.drop(['Valid', 'Task'], axis=1, inplace=True)

defog_data_valid2.head()

In [ ]:
# Apply the above function to our dataset
data_sample = create_features(defog_data_valid)

In [ ]:
# Create a function to reduce memory usage
def reduce_mem_usage(df):
    
    start_mem = df.memory_usage().sum() / 1024**2
    print('Memory usage of dataframe is {:.2f} MB'.format(start_mem))

    for col in df.columns:
        col_type = df[col].dtype.name

        if col_type not in ['object', 'category', 'datetime64[ns, UTC]']:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)

    end_mem = df.memory_usage().sum() / 1024**2
    print('Memory usage after optimization is: {:.2f} MB'.format(end_mem))
   
    return df 

In [ ]:
data_sample

In [ ]:
%%time

# Apply the above function to our dataset
full_data = reduce_mem_usage(data_sample)

In [ ]:
# Create features and targets
drop_cols = ['Time', 'StartHesitation', 'Turn', 'Walking', 'Id', 'dataset']

X = full_data.drop(drop_cols, axis=1)
y = full_data["Turn"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

X_train.shape

In [ ]:
X.columns

In [ ]:
# Сreate function for testing models
def test_model(algorithm, X_train, y_train, X_test, y_test):
    
    model = algorithm()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    f1 = f1_score(y_test, y_pred)
    accuracy = accuracy_score(y_test, y_pred)

        
    return f1, accuracy

#### XGBoost

In [ ]:

%%time

f1_xgb, accuracy_xgb = test_model(XGBClassifier, X_train, y_train, X_test, y_test)

print(f"XGBClassifier | F1 score - {f1_xgb}, | Accuracy - {accuracy_xgb}")

#### LightBGM

In [1]:
%%time

f1_lgbm, accuracy_lgbm = test_model(LGBMClassifier, X_train, y_train, X_test, y_test)

print(f"LGBMClassifier | F1 score - {f1_lgbm}, | Accuracy - {accuracy_lgbm}") 

NameError: name 'test_model' is not defined

#### CatBoostClassifier

In [ ]:
%%time

f1_cb, accuracy_cb = test_model(CatBoostClassifier, X_train, y_train, X_test, y_test)

print(f"CatBoostClassifier | F1 score - {f1_cb}, | Accuracy - {accuracy_cb}")

In [ ]:
from sklearn.dummy import DummyClassifier

dummy_clf = DummyClassifier(strategy="most_frequent", random_state=42)
dummy_clf.fit(X_train, y_train)
dummy_clf.predict(X_train)
accuracy_dummy = dummy_clf.score(X_train, y_train)

print("DummyClassifier | accuracy" , accuracy_dummy)

In [ ]:
# Create summary table
index = ["CatBoostClassifier",
         "XGBClassifier", 
         "LGBMClassifier",
         "DummyClassifier",
        ]

data = {'F1 score':[round(f1_cb,5),
                    round(f1_xgb,5),
                    round(f1_lgbm,5),
                    "-"],
        'Accuracy':[round(accuracy_cb, 5),
                    round(accuracy_xgb ,5),
                    round(accuracy_lgbm, 5),
                    round(accuracy_dummy, 5)],
        
        }

pd.DataFrame(data=data, index=index)

### walking

In [ ]:
# Create features and targets
drop_cols = ['Time', 'StartHesitation', 'Turn', 'Walking', 'Id', 'dataset']

X = full_data.drop(drop_cols, axis=1)
y_walking = full_data["Walking"]

X_train_walk, X_test_walk, y_train_walk, y_test_walk = train_test_split(X, y_walking, test_size=0.3, random_state=42, stratify=y_walking)

X_train_walk.shape

In [ ]:

%%time

f1_xgb_walk, accuracy_xgb_walk = test_model(XGBClassifier, X_train_walk, y_train_walk, X_test_walk, y_test_walk)

print(f"XGBClassifier | F1 score - {f1_xgb_walk}, | Accuracy - {accuracy_xgb_walk}")

In [ ]:
%%time

f1_lgbm_walk, accuracy_lgbm_walk = test_model(LGBMClassifier, X_train_walk, y_train_walk, X_test_walk, y_test_walk)

print(f"LGBMClassifier | F1 score - {f1_lgbm_walk}, | Accuracy - {accuracy_lgbm_walk}") 

In [ ]:
%%time

f1_cb_walk, accuracy_cb_walk = test_model(CatBoostClassifier, X_train_walk, y_train_walk, X_test_walk, y_test_walk)

print(f"CatBoostClassifier | F1 score - {f1_cb_walk}, | Accuracy - {accuracy_cb_walk}")

In [ ]:

dummy_clf_walk = DummyClassifier(strategy="most_frequent", random_state=42)
dummy_clf_walk.fit(X_train_walk, y_train_walk)
dummy_clf_walk.predict(X_train_walk)
accuracy_dummy_walk = dummy_clf_walk.score(X_train_walk, y_train_walk)

print("DummyClassifier | accuracy" , accuracy_dummy_walk)

In [ ]:
# Create summary table
index = ["CatBoostClassifier",
         "XGBClassifier", 
         "LGBMClassifier",
         "DummyClassifier",
        ]

data = {'F1 score':[round(f1_cb_walk,5),
                    round(f1_xgb_walk,5),
                    round(f1_lgbm_walk,5),
                    "-"],
        'Accuracy':[round(accuracy_cb_walk, 5),
                    round(accuracy_xgb_walk ,5),
                    round(accuracy_lgbm_walk, 5),
                    round(accuracy_dummy_walk, 5)]
        }

pd.DataFrame(data=data, index=index)

### Start Hesisation

In [ ]:
# Create features and targets
drop_cols = ['Time', 'StartHesitation', 'Turn', 'Walking', 'Id', 'dataset']

X = full_data.drop(drop_cols, axis=1)
y_StartHesitation = full_data["StartHesitation"]

X_train_StartHesitation, X_test_StartHesitation, y_train_StartHesitation, y_test_StartHesitation = train_test_split(X, y_StartHesitation, test_size=0.3, random_state=42, stratify=y_StartHesitation)

X_train_StartHesitation.shape

In [ ]:

%%time

f1_xgb_StartHesitation, accuracy_xgb_StartHesitation = test_model(XGBClassifier, X_train_StartHesitation, y_train_StartHesitation, X_test_StartHesitation, y_test_StartHesitation)

print(f"XGBClassifier | F1 score - {f1_xgb_StartHesitation}, | Accuracy - {accuracy_xgb_StartHesitation}")

In [ ]:
%%time

f1_lgbm_StartHesitation, accuracy_lgbm_StartHesitation = test_model(LGBMClassifier, X_train_StartHesitation, y_train_StartHesitation, X_test_StartHesitation, y_test_StartHesitation)

print(f"LGBMClassifier | F1 score - {f1_lgbm_StartHesitation}, | Accuracy - {accuracy_lgbm_StartHesitation}") 

In [ ]:
%%time

f1_cb_StartHesitation, accuracy_cb_StartHesitation = test_model(CatBoostClassifier,X_train_StartHesitation, y_train_StartHesitation, X_test_StartHesitation, y_test_StartHesitation)

print(f"CatBoostClassifier | F1 score - {f1_cb_StartHesitation}, | Accuracy - {accuracy_cb_StartHesitation}")

In [ ]:
dummy_clf_StartHesitation = DummyClassifier(strategy="most_frequent", random_state=42)
dummy_clf_StartHesitation.fit(X_train_StartHesitation, y_train_StartHesitation)
dummy_clf_StartHesitation.predict(X_train_StartHesitation)
accuracy_dummy_StartHesitation = dummy_clf_StartHesitation.score(X_train_StartHesitation, y_train_StartHesitation)

print("DummyClassifier | accuracy" , accuracy_dummy_StartHesitation)

In [ ]:
# Create summary table
index = ["CatBoostClassifier",
         "XGBClassifier", 
         "LGBMClassifier",
         "DummyClassifier",
        ]

data = {'F1 score':[round(f1_cb_StartHesitation,5),
                    round(f1_xgb_StartHesitation,5),
                    round(f1_lgbm_StartHesitation,5),
                    "-"],
        'Accuracy':[round(accuracy_cb_StartHesitation, 5),
                    round(accuracy_xgb_StartHesitation ,5),
                    round(accuracy_lgbm_StartHesitation, 5),
                    round(accuracy_dummy_StartHesitation, 5)]
        }

pd.DataFrame(data=data, index=index)